# Focused flavonoid-inspired molecular generation for HbS candidate screening

This notebook contains the **generative and medicinal-chemistry portion** of the study:

1. assemble/curate a flavonoid-like molecular corpus;
2. perform scaffold-aware train/validation/test splitting;
3. train a compact decoder-only SMILES transformer;
4. generate de novo flavonoid-inspired molecules;
5. calculate validity, uniqueness, exact novelty, scaffold and descriptor-space summaries;
6. apply the predefined medicinal-chemistry filtering cascade; and
7. export **all** molecules that pass the filters for downstream structure-based drug discovery (SBDD).

The model is used as a focused upstream molecule-supply tool.


In [ ]:
# ============================================================
# 0. INSTALLATION CELL
# ============================================================
# Kaggle note:
# - Turn Internet ON if you want live ChEMBL/PubChem collection.
# - If Internet is OFF, the notebook will still work with fallback flavonoids
#   and/or files uploaded to /kaggle/input.
# - Runtime: GPU recommended, CPU works for a smaller run.

import sys, subprocess, importlib.util, os

def is_installed(pkg_name: str) -> bool:
    return importlib.util.find_spec(pkg_name) is not None

def pip_install(packages, critical=False):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet"] + packages
    try:
        subprocess.check_call(cmd)
        print("Installed/verified:", " ".join(packages))
    except Exception as e:
        msg = f"Package installation failed for {packages}: {e}"
        if critical:
            raise RuntimeError(msg)
        print("WARNING:", msg)

# RDKit is usually available in Kaggle. If missing, this can install it.
if not is_installed("rdkit"):
    pip_install(["rdkit"], critical=True)

# Optional but useful packages
if not is_installed("chembl_webresource_client"):
    pip_install(["chembl_webresource_client"], critical=False)

if not is_installed("pubchempy"):
    pip_install(["pubchempy"], critical=False)

if not is_installed("tqdm"):
    pip_install(["tqdm"], critical=False)

In [ ]:
# ============================================================
# 1. IMPORTS, REPRODUCIBILITY, AND DIRECTORY STRUCTURE
# ============================================================
import os, re, json, math, time, random, shutil, warnings, platform
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

from rdkit import Chem, RDLogger
from rdkit.Chem import (
    AllChem, Descriptors, rdMolDescriptors, QED, Draw, PandasTools, DataStructs
)
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
from rdkit import RDConfig

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

RDLogger.DisableLog("rdApp.*")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

CFG = {
    "PROJECT": "flavonoid_gpt_hbs_candidates",
    "ROOT": next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "data").exists() and (p / "models").exists()), Path.cwd().resolve()),
    "USER_DATA_DIR": Path(os.environ.get("FLAVOGPT_USER_DATA_DIR", "/kaggle/input")),

    # Data curation
    "MIN_HEAVY_ATOMS": 10,
    "MAX_HEAVY_ATOMS": 80,
    "MAX_SALTS": 1,
    "TRAIN_VAL_TEST": (0.80, 0.10, 0.10),
    "AUGMENT_RANDOM_SMILES": True,
    "N_AUG_PER_MOL": 5,

    # GPT training
    "BLOCK_SIZE": 128,
    "BATCH_SIZE": 64,
    "EPOCHS": 25,
    "LR": 3e-4,
    "EMBED_DIM": 256,
    "N_HEAD": 8,
    "N_LAYER": 6,
    "DROPOUT": 0.10,
    "GRAD_CLIP": 1.0,
    "EARLY_STOPPING_PATIENCE": 5,

    # Generation
    "N_TO_GENERATE": 30000,
    "GEN_BATCH_SIZE": 256,
    "TEMPERATURE": 0.80,
    "TOP_K": 50,
    "TOP_P": 0.95,
    "MAX_GEN_LEN": 120,

    # Medicinal chemistry filters
    "QED_MIN": 0.50,
    "SA_MAX": 6.0,
    "ALLOW_PAINS": False,
    "ALLOW_BRENK": False,
    "LIPINSKI_MAX_VIOLATIONS": 1,
    "MW_RANGE": (150, 650),
    "LOGP_RANGE": (-1.5, 6.0),
    "TPSA_MAX": 180,
    "ROT_BONDS_MAX": 12,

    # Candidate export
    # None = export every compound that passes all filters; no top-N truncation.
    "TOP_N_EXPORT": None
}

# Directory layout
DIRS = {
    "raw": CFG["ROOT"] / "data" / "raw",
    "curated": CFG["ROOT"] / "data" / "curated",
    "splits": CFG["ROOT"] / "data" / "splits",
    "models": CFG["ROOT"] / "models",
    "generation": CFG["ROOT"] / "outputs" / "generation",
    "filtered": CFG["ROOT"] / "outputs" / "filtered_candidates",
    "figures": CFG["ROOT"] / "outputs" / "figures",
    "scaffolds": CFG["ROOT"] / "outputs" / "scaffolds",
    "reports": CFG["ROOT"] / "outputs" / "reports",
    "sbdd": CFG["ROOT"] / "sbdd_candidates",
    "logs": CFG["ROOT"] / "logs",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

with open(DIRS["reports"] / "config.json", "w") as f:
    json.dump({k: str(v) if isinstance(v, Path) else v for k,v in CFG.items()}, f, indent=2)

print("Project root:", CFG["ROOT"])

In [ ]:
# ============================================================
# 2. CHEMISTRY UTILITIES
# ============================================================

def mol_from_smiles(smiles: str):
    if not isinstance(smiles, str) or not smiles.strip():
        return None
    try:
        return Chem.MolFromSmiles(smiles.strip())
    except Exception:
        return None

def largest_fragment_mol(mol):
    if mol is None:
        return None
    frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=True)
    if not frags:
        return None
    return max(frags, key=lambda m: m.GetNumHeavyAtoms())

def neutralize_atoms(mol):
    # Conservative neutralization adapted from common RDKit cookbook pattern.
    if mol is None:
        return None
    pattern = Chem.MolFromSmarts("[+1!h0!$([*]~[-1,-2,-3,-4]),-1!$([*]~[+1,+2,+3,+4])]")
    at_matches = mol.GetSubstructMatches(pattern)
    mol_rw = Chem.RWMol(mol)
    for idx_tuple in at_matches:
        atom = mol_rw.GetAtomWithIdx(idx_tuple[0])
        chg = atom.GetFormalCharge()
        hcount = atom.GetTotalNumHs()
        atom.SetFormalCharge(0)
        atom.SetNumExplicitHs(max(0, hcount - chg))
        atom.UpdatePropertyCache(False)
    try:
        Chem.SanitizeMol(mol_rw)
        return mol_rw.GetMol()
    except Exception:
        return mol

def standardize_smiles(smiles: str, keep_largest=True, neutralize=True):
    mol = mol_from_smiles(smiles)
    if mol is None:
        return None
    if keep_largest:
        mol = largest_fragment_mol(mol)
    if neutralize:
        mol = neutralize_atoms(mol)
    if mol is None:
        return None
    try:
        Chem.SanitizeMol(mol)
        return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
    except Exception:
        return None

def randomize_smiles(smiles: str, n=5):
    mol = mol_from_smiles(smiles)
    if mol is None:
        return []
    out = set()
    for _ in range(n * 5):
        try:
            rs = Chem.MolToSmiles(mol, canonical=False, doRandom=True, isomericSmiles=True)
            if rs and rs != smiles:
                out.add(rs)
        except Exception:
            pass
        if len(out) >= n:
            break
    return list(out)

def murcko_scaffold_smiles(smiles: str):
    mol = mol_from_smiles(smiles)
    if mol is None:
        return None
    try:
        scaf = MurckoScaffold.GetScaffoldForMol(mol)
        if scaf is None or scaf.GetNumAtoms() == 0:
            return None
        return Chem.MolToSmiles(scaf, canonical=True)
    except Exception:
        return None

def morgan_fp(mol, radius=2, n_bits=2048):
    if mol is None:
        return None
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)

def tanimoto_to_set(smiles, ref_fps):
    mol = mol_from_smiles(smiles)
    fp = morgan_fp(mol)
    if fp is None or not ref_fps:
        return np.nan
    sims = DataStructs.BulkTanimotoSimilarity(fp, ref_fps)
    return float(max(sims)) if sims else np.nan

In [ ]:
# ============================================================
# 3. SYNTHETIC ACCESSIBILITY SCORE
# ============================================================
# Uses RDKit Contrib sascorer when available. Falls back to a transparent proxy
# if the Kaggle environment lacks the contrib module.

SA_SCORE_SOURCE = "fallback_proxy"

try:
    import sys
    sa_path = os.path.join(RDConfig.RDContribDir, "SA_Score")
    if sa_path not in sys.path:
        sys.path.append(sa_path)
    import sascorer
    SA_SCORE_SOURCE = "rdkit_contrib_sascorer"
except Exception:
    sascorer = None

def fallback_sa_score(mol):
    if mol is None:
        return np.nan
    heavy = mol.GetNumHeavyAtoms()
    rings = rdMolDescriptors.CalcNumRings(mol)
    chiral = len(Chem.FindMolChiralCenters(mol, includeUnassigned=True))
    hetero = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() not in [1, 6])
    rot = Descriptors.NumRotatableBonds(mol)
    # Scale approximately into 1 to 10, where lower is easier.
    score = 1.5 + 0.025*heavy + 0.25*rings + 0.20*chiral + 0.03*hetero + 0.08*rot
    return float(max(1.0, min(10.0, score)))

def compute_sa_score(mol):
    if mol is None:
        return np.nan
    if sascorer is not None:
        try:
            return float(sascorer.calculateScore(mol))
        except Exception:
            return fallback_sa_score(mol)
    return fallback_sa_score(mol)

print("SA score source:", SA_SCORE_SOURCE)

In [ ]:
# ============================================================
# 4. FLAVONOID-DOMAIN SMARTS AND PROPERTY FUNCTIONS
# ============================================================
# These SMARTS are intentionally broad. They are not meant to perfectly classify
# every flavonoid subtype, but to enrich the complete curated reference corpus for flavonoid-like chemistry.

FLAVONOID_SMARTS = {
    "flavone_flavonol_core": "O=c1cc(-c2ccccc2)oc2ccccc12",
    "flavanone_core": "O=C1CC(Oc2ccccc2)Oc2ccccc12",
    "isoflavone_like": "O=c1c(-c2ccccc2)coc2ccccc12",
    "chalcone_like": "O=CC=Cc1ccccc1",
    "chromone_core": "O=c1ccoc2ccccc12",
    "coumarin_core": "O=c1oc2ccccc2cc1",
    "benzopyran_core": "c1ccc2occc2c1",
    "phenylbenzopyran_broad": "c1ccc(cc1)-c1oc2ccccc2cc1",
}

FLAVONOID_PATS = {k: Chem.MolFromSmarts(v) for k, v in FLAVONOID_SMARTS.items()}

def flavonoid_pattern_hits(mol):
    if mol is None:
        return []
    hits = []
    for name, patt in FLAVONOID_PATS.items():
        if patt is not None and mol.HasSubstructMatch(patt):
            hits.append(name)
    return hits

def compute_properties(smiles: str):
    mol = mol_from_smiles(smiles)
    if mol is None:
        return None
    try:
        props = {
            "canonical_smiles": Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True),
            "MolWt": float(Descriptors.MolWt(mol)),
            "LogP": float(Descriptors.MolLogP(mol)),
            "HBD": int(Descriptors.NumHDonors(mol)),
            "HBA": int(Descriptors.NumHAcceptors(mol)),
            "TPSA": float(rdMolDescriptors.CalcTPSA(mol)),
            "RotBonds": int(Descriptors.NumRotatableBonds(mol)),
            "HeavyAtoms": int(mol.GetNumHeavyAtoms()),
            "Rings": int(rdMolDescriptors.CalcNumRings(mol)),
            "AromaticRings": int(rdMolDescriptors.CalcNumAromaticRings(mol)),
            "QED": float(QED.qed(mol)),
            "SA_Score": float(compute_sa_score(mol)),
            "FormalCharge": int(Chem.GetFormalCharge(mol)),
            "MurckoScaffold": murcko_scaffold_smiles(smiles),
        }
        props["LipinskiViolations"] = int(
            (props["MolWt"] > 500) +
            (props["LogP"] > 5) +
            (props["HBD"] > 5) +
            (props["HBA"] > 10)
        )
        props["VeberViolations"] = int((props["RotBonds"] > 10) + (props["TPSA"] > 140))
        hits = flavonoid_pattern_hits(mol)
        props["FlavonoidPatternHits"] = ";".join(hits)
        props["HasFlavonoidPattern"] = bool(hits)
        return props
    except Exception:
        return None

In [ ]:
# ============================================================
# 5. PAINS AND BRENK ALERT FILTERS
# ============================================================
params_pains = FilterCatalogParams()
params_pains.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
PAINS_CATALOG = FilterCatalog(params_pains)

params_brenk = FilterCatalogParams()
params_brenk.AddCatalog(FilterCatalogParams.FilterCatalogs.BRENK)
BRENK_CATALOG = FilterCatalog(params_brenk)

def catalog_matches(mol, catalog):
    if mol is None:
        return []
    try:
        entries = catalog.GetMatches(mol)
        return [e.GetDescription() for e in entries]
    except Exception:
        return []

def add_alert_columns(df, smiles_col="canonical_smiles"):
    pains_flags, pains_desc = [], []
    brenk_flags, brenk_desc = [], []
    for smi in tqdm(df[smiles_col].tolist(), desc="Checking PAINS/Brenk"):
        mol = mol_from_smiles(smi)
        p = catalog_matches(mol, PAINS_CATALOG)
        b = catalog_matches(mol, BRENK_CATALOG)
        pains_flags.append(len(p) > 0)
        pains_desc.append("; ".join(p))
        brenk_flags.append(len(b) > 0)
        brenk_desc.append("; ".join(b))
    out = df.copy()
    out["PAINS"] = pains_flags
    out["PAINS_Alerts"] = pains_desc
    out["BRENK"] = brenk_flags
    out["BRENK_Alerts"] = brenk_desc
    return out

In [ ]:
# ============================================================
# 6. DATA COLLECTION: USER FILES, FALLBACK SET, CHemBL, PubChem
# ============================================================
# You may upload extra flavonoid files to Kaggle input. Supported columns:
# smiles, SMILES, canonical_smiles, Canonical_SMILES, isomeric_smiles.

def find_user_smiles_files(root=Path("/kaggle/input")):
    files = []
    if not root.exists():
        return files
    for ext in ["*.csv", "*.tsv", "*.txt", "*.smi"]:
        files.extend(root.rglob(ext))
    return files

def load_smiles_from_file(path: Path):
    smiles = []
    try:
        if path.suffix.lower() == ".smi":
            for line in open(path, "r", encoding="utf-8", errors="ignore"):
                part = line.strip().split()
                if part:
                    smiles.append(part[0])
        elif path.suffix.lower() == ".txt":
            for line in open(path, "r", encoding="utf-8", errors="ignore"):
                token = line.strip().split()[0] if line.strip() else ""
                if token:
                    smiles.append(token)
        else:
            sep = "\t" if path.suffix.lower() == ".tsv" else ","
            df = pd.read_csv(path, sep=sep)
            possible = ["canonical_smiles", "Canonical_SMILES", "smiles", "SMILES", "isomeric_smiles", "Isomeric_SMILES"]
            col = next((c for c in possible if c in df.columns), None)
            if col:
                smiles.extend(df[col].dropna().astype(str).tolist())
    except Exception as e:
        print(f"Could not load {path}: {e}")
    return smiles

# Fallback seed set of common flavonoids/flavonoid-like compounds.
# This prevents total failure when Kaggle internet is disabled.
FALLBACK_FLAVONOID_SMILES = {
    "quercetin": "O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12",
    "kaempferol": "O=c1c(O)c(-c2ccc(O)cc2)oc2cc(O)cc(O)c12",
    "apigenin": "O=c1cc(-c2ccc(O)cc2)oc2cc(O)cc(O)c12",
    "luteolin": "O=c1cc(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12", 
    "chrysin": "O=c1cc(-c2ccccc2)oc2cc(O)cc(O)c12",
    "galangin": "O=c1c(O)c(-c2ccccc2)oc2cc(O)cc(O)c12",
    "naringenin": "O=C1CC(c2ccc(O)cc2)Oc2cc(O)cc(O)c21",
    "hesperetin": "COc1cc(O)c(C2CC(=O)c3c(O)cc(O)cc3O2)cc1O",
    "eriodictyol": "O=C1CC(c2ccc(O)c(O)c2)Oc2cc(O)cc(O)c21",
    "taxifolin": "O=C1C(O)C(c2ccc(O)c(O)c2)Oc2cc(O)cc(O)c21",
    "genistein": "O=c1c(-c2ccc(O)cc2)coc2cc(O)cc(O)c12",
    "daidzein": "O=c1c(-c2ccc(O)cc2)coc2ccc(O)cc12",
    "biochanin_A": "COc1ccc(-c2coc3cc(O)cc(O)c3c2=O)cc1",
    "formononetin": "COc1ccc(-c2coc3cc(O)ccc3c2=O)cc1",
    "catechin": "Oc1cc(O)c2C(O)C(c3ccc(O)c(O)c3)Oc2c1",
    "epicatechin": "Oc1cc(O)c2C(O)C(c3ccc(O)c(O)c3)Oc2c1",
    "fisetin": "O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)ccc12",
    "morin": "O=c1c(O)c(-c2cc(O)c(O)c(O)c2)oc2cc(O)cc(O)c12",
    "myricetin": "O=c1c(O)c(-c2cc(O)c(O)c(O)c2)oc2cc(O)cc(O)c12",
    "rutin_aglycone_quercetin": "O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12",
    "baicalein": "O=c1cc(-c2ccccc2)oc2cc(O)c(O)c(O)c12",
    "wogonin": "COc1c(O)cc2oc(-c3ccccc3)cc(=O)c2c1O",
    "pinocembrin": "O=C1CC(c2ccccc2)Oc2cc(O)cc(O)c21",
    "alpinetin": "COc1cc2oc(c3ccccc3)cc(=O)c2cc1O",
    "sakuranetin": "COc1ccc(C2CC(=O)c3c(O)cc(O)cc3O2)cc1",
    "liquiritigenin": "O=C1CC(c2ccc(O)cc2)Oc2cc(O)ccc21",
    "isoliquiritigenin": "O=C(/C=C/c1ccc(O)cc1)c1ccc(O)cc1",
    "butein": "O=C(/C=C/c1ccc(O)cc1)c1cc(O)cc(O)c1O",
    "phloretin": "O=C(CCc1ccc(O)cc1)c1c(O)cc(O)cc1O",
    "xanthohumol": "COc1cc(O)c(C/C=C(C)/C)c(O)c1C(=O)/C=C/c1ccc(O)cc1",
}

def collect_from_user_files():
    records = []
    files = find_user_smiles_files(CFG["USER_DATA_DIR"])
    for f in files:
        smi_list = load_smiles_from_file(f)
        for smi in smi_list:
            records.append({"source": "user_file", "source_detail": str(f), "smiles": smi})
    return pd.DataFrame(records)

def collect_from_chembl():
    records = []
    try:
        from chembl_webresource_client.new_client import new_client
        molecule = new_client.molecule
        queries = [
            "flavonoid", "flavone", "flavonol", "flavanone", "flavanol",
            "isoflavone", "chalcone", "coumarin", "chromone", "anthocyanidin",
            "quercetin", "kaempferol", "apigenin", "luteolin", "genistein",
            "naringenin", "catechin", "fisetin", "morin", "myricetin"
        ]
        seen = set()
        for q in tqdm(queries, desc="ChEMBL keyword search"):
            try:
                res = molecule.search(q)
                for r in res[:500]:
                    mol_struct = r.get("molecule_structures") or {}
                    smi = mol_struct.get("canonical_smiles")
                    chembl_id = r.get("molecule_chembl_id")
                    if smi and smi not in seen:
                        seen.add(smi)
                        records.append({"source": "chembl", "source_detail": q, "external_id": chembl_id, "smiles": smi})
            except Exception as e:
                print(f"ChEMBL query failed for {q}: {e}")
    except Exception as e:
        print("ChEMBL collection unavailable:", e)
    return pd.DataFrame(records)

def collect_from_pubchem_similarity(max_per_seed=40):
    records = []
    try:
        import pubchempy as pcp
        seeds = list(FALLBACK_FLAVONOID_SMILES.items())[:20]
        seen = set()
        for name, smi in tqdm(seeds, desc="PubChem similarity search"):
            try:
                comps = pcp.get_compounds(smi, namespace="smiles", searchtype="similarity", Threshold=80, MaxRecords=max_per_seed)
                for c in comps:
                    psmi = getattr(c, "canonical_smiles", None) or getattr(c, "isomeric_smiles", None)
                    cid = getattr(c, "cid", None)
                    if psmi and psmi not in seen:
                        seen.add(psmi)
                        records.append({"source": "pubchem_similarity", "source_detail": name, "external_id": cid, "smiles": psmi})
                time.sleep(0.2)
            except Exception as e:
                print(f"PubChem failed for {name}: {e}")
    except Exception as e:
        print("PubChem collection unavailable:", e)
    return pd.DataFrame(records)

fallback_df = pd.DataFrame([
    {"source": "fallback_seed", "source_detail": name, "external_id": name, "smiles": smi}
    for name, smi in FALLBACK_FLAVONOID_SMILES.items()
])

user_df = collect_from_user_files()
chembl_df = collect_from_chembl()
pubchem_df = collect_from_pubchem_similarity(max_per_seed=30)

raw_df = pd.concat([fallback_df, user_df, chembl_df, pubchem_df], ignore_index=True, sort=False)
raw_df.to_csv(DIRS["raw"] / "raw_collected_smiles.csv", index=False)

print("Raw molecules collected:", len(raw_df))
raw_df.head()

In [ ]:
# ============================================================
# 7. DATA CURATION AND FLAVONOID-DOMAIN ENRICHMENT
# ============================================================

def curate_raw_dataframe(raw_df):
    rows = []
    for _, row in tqdm(raw_df.iterrows(), total=len(raw_df), desc="Standardizing SMILES"):
        smi = row.get("smiles")
        can = standardize_smiles(smi)
        if can is None:
            continue
        mol = mol_from_smiles(can)
        if mol is None:
            continue
        heavy = mol.GetNumHeavyAtoms()
        if heavy < CFG["MIN_HEAVY_ATOMS"] or heavy > CFG["MAX_HEAVY_ATOMS"]:
            continue
        props = compute_properties(can)
        if props is None:
            continue
        out = row.to_dict()
        out.update(props)
        rows.append(out)
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("No valid molecules after curation. Upload a SMILES file or enable internet.")
    df = df.drop_duplicates("canonical_smiles").reset_index(drop=True)

    # Domain enrichment: accept molecules with a flavonoid/chromone/coumarin pattern
    # OR with >=2 aromatic rings and phenolic/HBA-rich flavonoid-like profile.
    df["DomainAccepted"] = (
        df["HasFlavonoidPattern"] |
        ((df["AromaticRings"] >= 2) & (df["HBA"] >= 2) & (df["Rings"] >= 2))
    )
    accepted = df[df["DomainAccepted"]].copy().reset_index(drop=True)
    return df, accepted

curated_all, curated = curate_raw_dataframe(raw_df)
curated_all.to_csv(DIRS["curated"] / "all_standardized_molecules.csv", index=False)
curated.to_csv(DIRS["curated"] / "curated_flavonoid_like_training_set.csv", index=False)

print("Standardized molecules:", len(curated_all))
print("Curated flavonoid-like molecules:", len(curated))
curated[["canonical_smiles","source","MolWt","LogP","QED","SA_Score","FlavonoidPatternHits"]].head()

In [ ]:
# ============================================================
# 8. CURATION SUMMARY FIGURES
# ============================================================

def save_hist(df, col, path, bins=30, xlabel=None):
    plt.figure(figsize=(6,4), dpi=150)
    plt.hist(df[col].dropna(), bins=bins)
    plt.xlabel(xlabel or col)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(path, dpi=300)
    plt.show()

for col in ["MolWt", "LogP", "TPSA", "QED", "SA_Score", "HeavyAtoms"]:
    save_hist(curated, col, DIRS["figures"] / f"curated_{col}_distribution.png")

source_counts = curated["source"].fillna("unknown").value_counts()
plt.figure(figsize=(7,4), dpi=150)
plt.bar(source_counts.index.astype(str), source_counts.values)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(DIRS["figures"] / "curated_sources.png", dpi=300)
plt.show()

print(curated.describe(include="all").T.head(20))

In [ ]:
# ============================================================
# 9. SCAFFOLD-AWARE SPLIT AND TRAINING SMILES AUGMENTATION
# ============================================================

def scaffold_aware_split(df, train_frac=0.8, val_frac=0.1, test_frac=0.1, seed=42):
    df = df.copy()
    df["MurckoScaffold"] = df["canonical_smiles"].apply(murcko_scaffold_smiles)
    scaffold_groups = df.groupby("MurckoScaffold", dropna=False).indices
    groups = list(scaffold_groups.values())
    random.Random(seed).shuffle(groups)

    n = len(df)
    n_train = int(train_frac * n)
    n_val = int(val_frac * n)

    train_idx, val_idx, test_idx = [], [], []
    for g in groups:
        target = (
            "train" if len(train_idx) < n_train else
            "val" if len(val_idx) < n_val else
            "test"
        )
        if target == "train":
            train_idx.extend(g)
        elif target == "val":
            val_idx.extend(g)
        else:
            test_idx.extend(g)
    return (
        df.iloc[train_idx].reset_index(drop=True),
        df.iloc[val_idx].reset_index(drop=True),
        df.iloc[test_idx].reset_index(drop=True),
    )

train_df, val_df, test_df = scaffold_aware_split(
    curated,
    train_frac=CFG["TRAIN_VAL_TEST"][0],
    val_frac=CFG["TRAIN_VAL_TEST"][1],
    test_frac=CFG["TRAIN_VAL_TEST"][2],
    seed=SEED
)

def augment_training_smiles(df, n_aug=5):
    rows = []
    for smi in tqdm(df["canonical_smiles"], desc="Random SMILES augmentation"):
        rows.append({"smiles": smi, "augmented": False})
        for rs in randomize_smiles(smi, n=n_aug):
            rows.append({"smiles": rs, "augmented": True})
    return pd.DataFrame(rows).drop_duplicates("smiles").reset_index(drop=True)

if CFG["AUGMENT_RANDOM_SMILES"]:
    train_smiles_df = augment_training_smiles(train_df, CFG["N_AUG_PER_MOL"])
else:
    train_smiles_df = pd.DataFrame({"smiles": train_df["canonical_smiles"], "augmented": False})

val_smiles_df = pd.DataFrame({"smiles": val_df["canonical_smiles"], "augmented": False})
test_smiles_df = pd.DataFrame({"smiles": test_df["canonical_smiles"], "augmented": False})

train_df.to_csv(DIRS["splits"] / "train_scaffold_split.csv", index=False)
val_df.to_csv(DIRS["splits"] / "val_scaffold_split.csv", index=False)
test_df.to_csv(DIRS["splits"] / "test_scaffold_split.csv", index=False)
train_smiles_df.to_csv(DIRS["splits"] / "train_smiles_augmented.csv", index=False)

print("Train molecules:", len(train_df), "Augmented train strings:", len(train_smiles_df))
print("Validation molecules:", len(val_df))
print("Test molecules:", len(test_df))

In [ ]:
# ============================================================
# 10. SMILES TOKENIZER FOR GPT
# ============================================================

# Preserve the exact regex used for the archived checkpoint.
import sys
if str(CFG["ROOT"] / "scripts") not in sys.path:
    sys.path.insert(0, str(CFG["ROOT"] / "scripts"))
from smiles_tokenization import TOKEN_REGEX

SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<unk>"]

def tokenize_smiles(smiles):
    return TOKEN_REGEX.findall(smiles)

def build_vocab(smiles_list):
    counter = Counter()
    for smi in smiles_list:
        counter.update(tokenize_smiles(smi))
    tokens = SPECIAL_TOKENS + sorted(counter.keys())
    stoi = {tok:i for i,tok in enumerate(tokens)}
    itos = {i:tok for tok,i in stoi.items()}
    return stoi, itos, counter

stoi, itos, tok_counter = build_vocab(train_smiles_df["smiles"].tolist())
vocab_size = len(stoi)
PAD, BOS, EOS, UNK = [stoi[t] for t in SPECIAL_TOKENS]

def encode_smiles(smiles, max_len=CFG["BLOCK_SIZE"]):
    toks = ["<bos>"] + tokenize_smiles(smiles) + ["<eos>"]
    ids = [stoi.get(t, UNK) for t in toks]
    ids = ids[:max_len]
    if len(ids) < max_len:
        ids += [PAD] * (max_len - len(ids))
    return ids

def decode_ids(ids):
    toks = []
    for i in ids:
        tok = itos.get(int(i), "<unk>")
        if tok in ["<bos>", "<pad>"]:
            continue
        if tok == "<eos>":
            break
        toks.append(tok)
    return "".join(toks)

tokenizer_info = {
    "stoi": stoi,
    "itos": {str(k):v for k,v in itos.items()},
    "vocab_size": vocab_size,
    "block_size": CFG["BLOCK_SIZE"],
    "special_tokens": SPECIAL_TOKENS,
}
with open(DIRS["models"] / "smiles_tokenizer.json", "w") as f:
    json.dump(tokenizer_info, f, indent=2)

print("Vocabulary size:", vocab_size)
print("Most common tokens:", tok_counter.most_common(20))

In [ ]:
# ============================================================
# 11. DATASETS AND DATALOADERS
# ============================================================

class SmilesDataset(Dataset):
    def __init__(self, smiles_list, max_len):
        self.smiles = list(smiles_list)
        self.max_len = max_len

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        ids = encode_smiles(self.smiles[idx], self.max_len)
        x = torch.tensor(ids[:-1], dtype=torch.long)
        y = torch.tensor(ids[1:], dtype=torch.long)
        return x, y

train_ds = SmilesDataset(train_smiles_df["smiles"].tolist(), CFG["BLOCK_SIZE"])
val_ds = SmilesDataset(val_smiles_df["smiles"].tolist(), CFG["BLOCK_SIZE"])

train_loader = DataLoader(train_ds, batch_size=CFG["BATCH_SIZE"], shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=CFG["BATCH_SIZE"], shuffle=False, num_workers=2, pin_memory=True)

print("Train batches:", len(train_loader), "Val batches:", len(val_loader))

In [ ]:
# ============================================================
# 12. DECODER-ONLY SMILES GPT MODEL
# ============================================================

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(block_size, block_size)).view(1,1,block_size,block_size)
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1,2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1,2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1,2)
        att = (q @ k.transpose(-2,-1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:,:,:T,:T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)
        y = att @ v
        y = y.transpose(1,2).contiguous().view(B,T,C)
        y = self.resid_drop(self.proj(y))
        return y

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.GELU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class SmilesGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd=256, n_head=8, n_layer=6, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        if T > self.block_size:
            raise ValueError("Sequence length exceeds block size")
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device).unsqueeze(0)
        x = self.drop(self.token_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1), ignore_index=PAD)
        return logits, loss

model = SmilesGPT(
    vocab_size=vocab_size,
    block_size=CFG["BLOCK_SIZE"]-1,
    n_embd=CFG["EMBED_DIM"],
    n_head=CFG["N_HEAD"],
    n_layer=CFG["N_LAYER"],
    dropout=CFG["DROPOUT"]
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

In [ ]:
# ============================================================
# 13. TRAINING LOOP WITH CHECKPOINTING
# ============================================================

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["LR"], weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, CFG["EPOCHS"]))

def run_epoch(loader, train=True):
    model.train(train)
    total_loss, n_batches = 0.0, 0
    for x, y in tqdm(loader, leave=False):
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        with torch.set_grad_enabled(train):
            logits, loss = model(x, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["GRAD_CLIP"])
                optimizer.step()
        total_loss += float(loss.item())
        n_batches += 1
    return total_loss / max(1, n_batches)

history = []
best_val = float("inf")
patience = 0
best_path = DIRS["models"] / "best_smiles_gpt.pt"

for epoch in range(1, CFG["EPOCHS"] + 1):
    train_loss = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader, train=False)
    scheduler.step()
    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_ppl": float(math.exp(min(train_loss, 20))),
        "val_ppl": float(math.exp(min(val_loss, 20))),
        "lr": optimizer.param_groups[0]["lr"]
    }
    history.append(row)
    print(f"Epoch {epoch:03d} | train loss {train_loss:.4f} | val loss {val_loss:.4f} | val ppl {row['val_ppl']:.2f}")

    if val_loss < best_val:
        best_val = val_loss
        patience = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "config": CFG,
            "vocab_size": vocab_size,
            "stoi": stoi,
            "itos": itos,
            "epoch": epoch,
            "val_loss": val_loss
        }, best_path)
        print("  Saved new best checkpoint.")
    else:
        patience += 1
        if patience >= CFG["EARLY_STOPPING_PATIENCE"]:
            print("Early stopping triggered.")
            break

hist_df = pd.DataFrame(history)
hist_df.to_csv(DIRS["models"] / "training_history.csv", index=False)

plt.figure(figsize=(6,4), dpi=150)
plt.plot(hist_df["epoch"], hist_df["train_loss"], label="Train")
plt.plot(hist_df["epoch"], hist_df["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.legend()
plt.tight_layout()
plt.savefig(DIRS["figures"] / "smiles_gpt_training_curve.png", dpi=300)
plt.show()

In [ ]:
# ============================================================
# 14. MOLECULE GENERATION
# ============================================================

# Load best model for generation
# This checkpoint was created by this project and contains configuration Paths.
# Explicit loading avoids the PyTorch >=2.6 weights_only default mismatch.
ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

def top_k_top_p_filtering(logits, top_k=0, top_p=1.0):
    logits = logits.clone()
    if top_k and top_k > 0:
        top_k = min(top_k, logits.size(-1))
        values, _ = torch.topk(logits, top_k)
        min_values = values[:, -1].unsqueeze(1)
        logits[logits < min_values] = -float("inf")
    if top_p and top_p < 1.0:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[:, 1:] = sorted_indices_to_remove[:, :-1].clone()
        sorted_indices_to_remove[:, 0] = 0
        for b in range(logits.size(0)):
            indices_to_remove = sorted_indices[b, sorted_indices_to_remove[b]]
            logits[b, indices_to_remove] = -float("inf")
    return logits

@torch.no_grad()
def generate_smiles_batch(batch_size=128, max_len=120, temperature=1.0, top_k=50, top_p=0.95):
    idx = torch.full((batch_size, 1), BOS, dtype=torch.long, device=DEVICE)
    finished = torch.zeros(batch_size, dtype=torch.bool, device=DEVICE)

    for _ in range(max_len):
        idx_cond = idx[:, -model.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / max(temperature, 1e-6)
        logits[:, PAD] = -float("inf")
        logits = top_k_top_p_filtering(logits, top_k=top_k, top_p=top_p)
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
        finished |= (next_id.squeeze(1) == EOS)
        if finished.all():
            break
    return [decode_ids(row.tolist()) for row in idx]

generated = []
n_batches = math.ceil(CFG["N_TO_GENERATE"] / CFG["GEN_BATCH_SIZE"])
for _ in tqdm(range(n_batches), desc="Generating SMILES"):
    batch = generate_smiles_batch(
        batch_size=CFG["GEN_BATCH_SIZE"],
        max_len=CFG["MAX_GEN_LEN"],
        temperature=CFG["TEMPERATURE"],
        top_k=CFG["TOP_K"],
        top_p=CFG["TOP_P"]
    )
    generated.extend(batch)

generated = generated[:CFG["N_TO_GENERATE"]]
gen_raw_df = pd.DataFrame({"generated_smiles": generated})
gen_raw_df.to_csv(DIRS["generation"] / "Table_S9_raw_generated_smiles.csv", index=False)
print("Generated strings:", len(gen_raw_df))
gen_raw_df.head()

In [ ]:
# ============================================================
# 15. GENERATION VALIDITY, UNIQUENESS, NOVELTY
# ============================================================

reference_set = set(curated["canonical_smiles"].tolist())

gen_records = []
for smi in tqdm(gen_raw_df["generated_smiles"], desc="Validating generated SMILES"):
    can = standardize_smiles(smi)
    valid = can is not None
    rec = {"generated_smiles": smi, "valid": valid, "canonical_smiles": can}
    gen_records.append(rec)

gen_eval_df = pd.DataFrame(gen_records)
valid_df = gen_eval_df[gen_eval_df["valid"]].drop_duplicates("canonical_smiles").reset_index(drop=True)
valid_df["novel_exact"] = ~valid_df["canonical_smiles"].isin(reference_set)

validity = gen_eval_df["valid"].mean()
uniqueness_among_valid = len(valid_df) / max(1, gen_eval_df["valid"].sum())
novelty_among_unique_valid = valid_df["novel_exact"].mean() if len(valid_df) else 0

metrics = {
    "n_generated": int(len(gen_eval_df)),
    "n_valid": int(gen_eval_df["valid"].sum()),
    "n_unique_valid": int(len(valid_df)),
    "n_novel_unique_valid": int(valid_df["novel_exact"].sum()),
    "validity": float(validity),
    "uniqueness_among_valid": float(uniqueness_among_valid),
    "novelty_among_unique_valid": float(novelty_among_unique_valid),
}
with open(DIRS["generation"] / "generation_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

valid_df.to_csv(DIRS["generation"] / "Table_S10_valid_unique_generated_molecules.csv", index=False)

plt.figure(figsize=(6,4), dpi=150)
plt.bar(["Validity", "Uniqueness", "Novelty"], [validity, uniqueness_among_valid, novelty_among_unique_valid])
plt.ylim(0,1)
plt.ylabel("Fraction")
plt.tight_layout()
plt.savefig(DIRS["figures"] / "generation_validity_uniqueness_novelty.png", dpi=300)
plt.show()

metrics

In [ ]:
# ============================================================
# 16. PROPERTY CALCULATION FOR GENERATED MOLECULES
# ============================================================

prop_rows = []
for smi in tqdm(valid_df["canonical_smiles"], desc="Computing generated molecule properties"):
    props = compute_properties(smi)
    if props is not None:
        prop_rows.append(props)

gen_props = pd.DataFrame(prop_rows).drop_duplicates("canonical_smiles").reset_index(drop=True)
gen_props["novel_exact"] = ~gen_props["canonical_smiles"].isin(reference_set)

# Similarity to training molecules for novelty/context
reference_fps = [morgan_fp(mol_from_smiles(s)) for s in curated["canonical_smiles"]]
reference_fps = [fp for fp in reference_fps if fp is not None]
gen_props["MaxTrainTanimoto"] = [
    tanimoto_to_set(smi, reference_fps) for smi in tqdm(gen_props["canonical_smiles"], desc="Reference-corpus similarity")
]

gen_props.to_csv(DIRS["generation"] / "Table_S11_generated_properties_before_filters.csv", index=False)
print("Generated molecules with properties:", len(gen_props))
gen_props.head()

In [ ]:
# ============================================================
# 17. MEDICINAL CHEMISTRY FILTERING
# ============================================================

filtered = add_alert_columns(gen_props, smiles_col="canonical_smiles")

mw_min, mw_max = CFG["MW_RANGE"]
logp_min, logp_max = CFG["LOGP_RANGE"]

filtered["Pass_MW"] = filtered["MolWt"].between(mw_min, mw_max)
filtered["Pass_LogP"] = filtered["LogP"].between(logp_min, logp_max)
filtered["Pass_TPSA"] = filtered["TPSA"] <= CFG["TPSA_MAX"]
filtered["Pass_RotBonds"] = filtered["RotBonds"] <= CFG["ROT_BONDS_MAX"]
filtered["Pass_Lipinski"] = filtered["LipinskiViolations"] <= CFG["LIPINSKI_MAX_VIOLATIONS"]
filtered["Pass_QED"] = filtered["QED"] >= CFG["QED_MIN"]
filtered["Pass_SA"] = filtered["SA_Score"] <= CFG["SA_MAX"]
filtered["Pass_PAINS"] = ~filtered["PAINS"] if not CFG["ALLOW_PAINS"] else True
filtered["Pass_BRENK"] = ~filtered["BRENK"] if not CFG["ALLOW_BRENK"] else True
filtered["Pass_Novel"] = filtered["novel_exact"] == True
filtered["Pass_Domain"] = filtered["DomainAccepted"] if "DomainAccepted" in filtered.columns else (
    filtered["HasFlavonoidPattern"] | ((filtered["AromaticRings"] >= 2) & (filtered["HBA"] >= 2))
)

filter_cols = [
    "Pass_Novel", "Pass_Domain", "Pass_MW", "Pass_LogP", "Pass_TPSA", "Pass_RotBonds",
    "Pass_Lipinski", "Pass_QED", "Pass_SA", "Pass_PAINS", "Pass_BRENK"
]
filtered["Pass_All_Filters"] = filtered[filter_cols].all(axis=1)

# No composite ranking is applied.
# All molecules that pass the predefined medicinal-chemistry filters are retained
# and advanced to SBDD preparation.
passed = filtered[filtered["Pass_All_Filters"]].copy().reset_index(drop=True)
passed["Candidate_ID"] = [f"FGHBS_{i+1:05d}" for i in range(len(passed))]

filtered.to_csv(DIRS["filtered"] / "generated_molecules_all_filter_annotations.csv", index=False)
passed.to_csv(DIRS["filtered"] / "Table_S12_all_2371_candidates_passing_filters.csv", index=False)

summary = []
current_mask = pd.Series(True, index=filtered.index)
for col in filter_cols:
    before = int(current_mask.sum())
    current_mask &= filtered[col]
    after = int(current_mask.sum())
    summary.append({"filter": col, "before": before, "after": after, "removed": before-after})
filter_summary = pd.DataFrame(summary)
filter_summary.to_csv(DIRS["filtered"] / "Table_S13_filtering_summary.csv", index=False)

plt.figure(figsize=(8,4), dpi=150)
plt.bar(filter_summary["filter"], filter_summary["after"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Molecules remaining")
plt.tight_layout()
plt.savefig(DIRS["figures"] / "filtering_cascade.png", dpi=300)
plt.show()

print("Generated molecules before filters:", len(filtered))
print("Passed all filters:", len(passed))
print("All passed compounds retained for SBDD; no ranking or top-N truncation applied.")
filter_summary


In [ ]:
# ============================================================
# 18. PROPERTY DISTRIBUTION PLOTS AFTER FILTERING
# ============================================================

plot_cols = ["MolWt", "LogP", "TPSA", "QED", "SA_Score", "RotBonds", "MaxTrainTanimoto"]
for col in plot_cols:
    if col not in filtered.columns:
        continue
    plt.figure(figsize=(6,4), dpi=150)
    plt.hist(filtered[col].dropna(), bins=30, alpha=0.6, label="All generated")
    if len(passed) > 0:
        plt.hist(passed[col].dropna(), bins=30, alpha=0.6, label="SBDD-ready")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.legend()
    plt.tight_layout()
    plt.savefig(DIRS["figures"] / f"generated_vs_passed_{col}.png", dpi=300)
    plt.show()

In [ ]:
# ============================================================
# 19. BEMIS-MURCKO SCAFFOLD ANALYSIS
# ============================================================

def scaffold_summary(df, label):
    tmp = df.copy()
    tmp["MurckoScaffold"] = tmp["canonical_smiles"].apply(murcko_scaffold_smiles)
    counts = tmp["MurckoScaffold"].fillna("NO_SCAFFOLD").value_counts().reset_index()
    counts.columns = ["MurckoScaffold", "Count"]
    counts["Dataset"] = label
    return counts

reference_scaf = scaffold_summary(curated, "Curated reference corpus")
gen_scaf = scaffold_summary(gen_props, "Generated valid unique")
passed_scaf = scaffold_summary(passed, "Passed candidates")

reference_scaf_set = set(reference_scaf["MurckoScaffold"]) - {"NO_SCAFFOLD"}
passed_scaf["NovelScaffoldVsReference"] = ~passed_scaf["MurckoScaffold"].isin(reference_scaf_set)

scaf_all = pd.concat([reference_scaf, gen_scaf, passed_scaf], ignore_index=True, sort=False)
scaf_all.to_csv(DIRS["scaffolds"] / "scaffold_summary_all_sets.csv", index=False)
passed_scaf.to_csv(DIRS["scaffolds"] / "passed_candidate_scaffolds.csv", index=False)

top_passed_scaf = passed_scaf.head(15)
plt.figure(figsize=(8,4), dpi=150)
plt.bar([f"S{i+1}" for i in range(len(top_passed_scaf))], top_passed_scaf["Count"])
plt.xlabel("Top passed scaffold")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(DIRS["figures"] / "top_passed_scaffold_frequencies.png", dpi=300)
plt.show()

# Draw top scaffold structures
top_scafs = [s for s in top_passed_scaf["MurckoScaffold"].tolist() if s != "NO_SCAFFOLD"][:12]
mols = [mol_from_smiles(s) for s in top_scafs]
legends = [f"S{i+1} | n={int(top_passed_scaf.iloc[i]['Count'])}" for i in range(len(mols))]
if mols:
    img = Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(250,200), legends=legends)
    img.save(str(DIRS["figures"] / "top_passed_scaffold_structures.png"))
    display(img)

print("Unique reference-corpus scaffolds:", len(reference_scaf))
print("Unique generated scaffolds:", len(gen_scaf))
print("Unique passed scaffolds:", len(passed_scaf))
print("Novel passed scaffolds vs reference corpus:", int(passed_scaf.get("NovelScaffoldVsReference", pd.Series(dtype=bool)).sum()))

In [ ]:
# ============================================================
# 20. CHEMICAL SPACE ANALYSIS: REFERENCE VS GENERATED VS SBDD-READY
# ============================================================

def descriptors_for_space(df, label, max_n=2000):
    use = df.copy()
    if len(use) > max_n:
        use = use.sample(max_n, random_state=SEED)
    cols = ["MolWt","LogP","TPSA","HBD","HBA","RotBonds","Rings","AromaticRings","QED","SA_Score"]
    out = use[["canonical_smiles"] + cols].copy()
    out["Set"] = label
    return out

space_df = pd.concat([
    descriptors_for_space(curated, "Curated reference"),
    descriptors_for_space(gen_props, "Generated"),
    descriptors_for_space(passed, "SBDD-ready")
], ignore_index=True)

space_cols = ["MolWt","LogP","TPSA","HBD","HBA","RotBonds","Rings","AromaticRings","QED","SA_Score"]
X = space_df[space_cols].replace([np.inf,-np.inf], np.nan).fillna(space_df[space_cols].median())
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2, random_state=SEED)
coords = pca.fit_transform(X_scaled)
space_df["PC1"] = coords[:,0]
space_df["PC2"] = coords[:,1]
space_df.to_csv(DIRS["generation"] / "chemical_space_coordinates.csv", index=False)

plt.figure(figsize=(7,5), dpi=150)
for label in ["Curated reference", "Generated", "SBDD-ready"]:
    sub = space_df[space_df["Set"] == label]
    plt.scatter(sub["PC1"], sub["PC2"], s=12, alpha=0.65, label=label)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.legend()
plt.tight_layout()
plt.savefig(DIRS["figures"] / "chemical_space_pca_reference_generated_sbdd_ready.png", dpi=300)
plt.show()

# Optional t-SNE if sample is small enough
if len(space_df) <= 4000:
    perplexity = min(30, max(5, len(space_df)//20))
    tsne = TSNE(n_components=2, perplexity=perplexity, learning_rate="auto", init="pca", random_state=SEED)
    ts = tsne.fit_transform(X_scaled)
    space_df["tSNE1"] = ts[:,0]
    space_df["tSNE2"] = ts[:,1]
    plt.figure(figsize=(7,5), dpi=150)
    for label in ["Curated reference", "Generated", "SBDD-ready"]:
        sub = space_df[space_df["Set"] == label]
        plt.scatter(sub["tSNE1"], sub["tSNE2"], s=12, alpha=0.65, label=label)
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.legend()
    plt.tight_layout()
    plt.savefig(DIRS["figures"] / "chemical_space_tsne_reference_generated_sbdd_ready.png", dpi=300)
    plt.show()

In [ ]:
# ============================================================
# 21. FINAL EXPORT FOR SBDD
# ============================================================

if len(passed) == 0:
    raise RuntimeError("No compounds passed all filters. Consider relaxing QED_MIN, SA_MAX, or alert filters.")

# Export ALL compounds that passed the filtering cascade.
# No composite score, sorting, or top-N truncation is applied.
export_df = passed.copy().reset_index(drop=True)

export_cols = [
    "Candidate_ID", "canonical_smiles", "QED", "SA_Score",
    "MolWt", "LogP", "TPSA", "HBD", "HBA", "RotBonds", "Rings",
    "AromaticRings", "LipinskiViolations", "VeberViolations", "MaxTrainTanimoto",
    "MurckoScaffold", "FlavonoidPatternHits", "PAINS", "BRENK",
    "PAINS_Alerts", "BRENK_Alerts"
]
export_cols = [c for c in export_cols if c in export_df.columns]

# CSV and Excel exports
export_df[export_cols].to_csv(DIRS["sbdd"] / "final_candidates_for_sbdd_all_passed.csv", index=False)
try:
    export_df[export_cols].to_excel(DIRS["sbdd"] / "final_candidates_for_sbdd_all_passed.xlsx", index=False)
except Exception as e:
    print("Excel export skipped because openpyxl/xlsxwriter may be unavailable:", repr(e))

# SMI export for downstream SBDD workflows
with open(DIRS["sbdd"] / "final_candidates_for_sbdd_all_passed.smi", "w") as f:
    for _, r in export_df.iterrows():
        f.write(f"{r['canonical_smiles']}\t{r['Candidate_ID']}\n")

# SDF export with 2D coordinates
mols = []
for _, r in export_df.iterrows():
    mol = mol_from_smiles(r["canonical_smiles"])
    if mol is None:
        continue
    mol.SetProp("_Name", str(r["Candidate_ID"]))
    for col in export_cols:
        if col in r and pd.notna(r[col]):
            mol.SetProp(str(col), str(r[col]))
    try:
        AllChem.Compute2DCoords(mol)
    except Exception:
        pass
    mols.append(mol)

writer = Chem.SDWriter(str(DIRS["sbdd"] / "final_candidates_for_sbdd_all_passed.sdf"))
for mol in mols:
    writer.write(mol)
writer.close()

# Draw first 20 retained structures for quick visual inspection only.
preview_mols = mols[:20]
if preview_mols:
    legends = [export_df.iloc[i]["Candidate_ID"] for i in range(len(preview_mols))]
    img = Draw.MolsToGridImage(preview_mols, molsPerRow=5, subImgSize=(220,180), legends=legends)
    img.save(str(DIRS["figures"] / "preview_20_sbdd_ready_candidates.png"))

manifest = {
    "generated_requested": CFG["N_TO_GENERATE"],
    "generated_unique_valid_after_generation_stage": int(len(gen_props)) if "gen_props" in globals() else None,
    "passed_all_filters_exported": int(len(export_df)),
    "export_policy": "all_passed_no_ranking_no_top_n_truncation",
    "files": {
        "csv": str(DIRS["sbdd"] / "final_candidates_for_sbdd_all_passed.csv"),
        "xlsx": str(DIRS["sbdd"] / "final_candidates_for_sbdd_all_passed.xlsx"),
        "smi": str(DIRS["sbdd"] / "final_candidates_for_sbdd_all_passed.smi"),
        "sdf": str(DIRS["sbdd"] / "final_candidates_for_sbdd_all_passed.sdf"),
    }
}
with open(DIRS["sbdd"] / "sbdd_export_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Generated molecules requested:", CFG["N_TO_GENERATE"])
print("Exported ALL passed compounds for SBDD:", len(export_df))
print("SBDD-ready files saved in:", DIRS["sbdd"])


In [ ]:
# ============================================================
# 22. REPRODUCIBILITY REPORT AND SESSION ARCHIVE
# ============================================================

report = {
    "project": CFG["PROJECT"],
    "timestamp": datetime.now().isoformat(),
    "platform": platform.platform(),
    "python": platform.python_version(),
    "device": str(DEVICE),
    "torch_version": torch.__version__,
    "rdkit_version": Chem.rdBase.rdkitVersion,
    "sa_score_source": SA_SCORE_SOURCE,
    "raw_collected": int(len(raw_df)),
    "curated_reference_corpus": int(len(curated)),
    "train_molecules": int(len(train_df)),
    "val_molecules": int(len(val_df)),
    "test_molecules": int(len(test_df)),
    "generated_strings": int(len(gen_eval_df)),
    "valid_generated": int(gen_eval_df["valid"].sum()),
    "unique_valid_generated": int(len(gen_props)),
    "passed_filters": int(len(passed)),
    "exported_candidates": int(len(export_df)),
    "generation_metrics": metrics,
    "best_val_loss": float(best_val),
    "config": {k: str(v) if isinstance(v, Path) else v for k,v in CFG.items()}
}
with open(DIRS["reports"] / "run_summary.json", "w") as f:
    json.dump(report, f, indent=2)

# Human-readable summary
summary_lines = [
    "# Run Summary",
    f"- Project: {CFG['PROJECT']}",
    f"- Timestamp: {report['timestamp']}",
    f"- Device: {report['device']}",
    f"- Curated reference-corpus molecules: {report['curated_reference_corpus']}",
    f"- Generated strings: {report['generated_strings']}",
    f"- Valid generated: {report['valid_generated']}",
    f"- Unique valid generated: {report['unique_valid_generated']}",
    f"- Passed filters: {report['passed_filters']}",
    f"- Exported candidates: {report['exported_candidates']}",
    "",
    "## Key output files",
    f"- Final candidates CSV: `{DIRS['sbdd'] / 'final_candidates_for_sbdd_all_passed.csv'}`",
    f"- Final candidates SMI: `{DIRS['sbdd'] / 'final_candidates_for_sbdd_all_passed.smi'}`",
    f"- Final candidates SDF: `{DIRS['sbdd'] / 'final_candidates_for_sbdd_all_passed.sdf'}`",
    f"- Best GPT model: `{best_path}`",
    f"- Tokenizer: `{DIRS['models'] / 'smiles_tokenizer.json'}`",
]
with open(DIRS["reports"] / "RUN_SUMMARY.md", "w") as f:
    f.write("\n".join(summary_lines))

print("Saved reproducibility report:", DIRS["reports"] / "RUN_SUMMARY.md")



Use `notebooks/02_heldout_scaffold_evaluation.ipynb`  to reproduce the held-out scaffold test metrics reported in the manuscript.
